In [1]:
from pyspark.sql import SparkSession

# 스파크 세션 생성
spark = SparkSession.builder \
    .appName("SparkTest") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.0


In [2]:
# 1부터 10까지의 숫자가 담긴 파이썬 리스트를 RDD로 변환
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd = spark.sparkContext.parallelize(data)

print("RDD 타입:", type(rdd))

RDD 타입: <class 'pyspark.rdd.RDD'>


In [3]:
# 각 요소에 2를 곱하기 (Transformation)
mapped_rdd = rdd.map(lambda x: x * 2)

# 10보다 큰 수만 필터링하기 (Transformation)
filtered_rdd = mapped_rdd.filter(lambda x: x > 10)

In [4]:
# 1. collect(): RDD의 모든 데이터를 드라이버 노드로 가져와 파이썬 리스트로 출력
print("결과 데이터:", filtered_rdd.collect())

# 2. count(): 조건에 맞는 데이터 개수 확인
print("데이터 개수:", filtered_rdd.count())

# 3. reduce(): 모든 요소를 합치거나 특정 연산 수행 (여기서는 모두 더하기 / 전체 누적이 핵심)
sum_result = rdd.reduce(lambda a, b: a + b)
print("1부터 10까지의 총합:", sum_result)

결과 데이터: [12, 14, 16, 18, 20]
데이터 개수: 5
1부터 10까지의 총합: 55


In [5]:
# 텍스트 데이터 준비
text_data = [
    "아이해브 아이브",
    "안유진 가을 레이 장원영 리즈 이서",
    "나오이 레이" ,
    "장원영 원영 장원영 이서"
]

text_rdd = spark.sparkContext.parallelize(text_data)

# 1. 문장을 띄어쓰기 기준으로 쪼개기 (flatMap)
words_rdd = text_rdd.flatMap(lambda line: line.split(" "))

# 2. 각 단어를 (단어, 1) 형태의 튜플로 만들기 (map)
word_tuples = words_rdd.map(lambda word: (word, 1))

# 3. 같은 단어끼리 더하기 (reduceByKey)
word_counts = word_tuples.reduceByKey(lambda a, b: a + b)

# 4. 결과 출력 (collect)
print("단어 빈도수:", word_counts.collect())


단어 빈도수: [('아이브', 1), ('가을', 1), ('아이해브', 1), ('안유진', 1), ('레이', 2), ('장원영', 3), ('리즈', 1), ('이서', 2), ('나오이', 1), ('원영', 1)]


In [6]:
# 1. 튜플 형태의 RDD를 (단어, 빈도수) 구조화된 DataFrame으로 변환
df = word_counts.toDF(["word", "count"])

# 2. 스키마 구조 확인
df.printSchema()

# 3. 데이터프레임 내용 출력 (쇼우)
df.show()

root
 |-- word: string (nullable = true)
 |-- count: long (nullable = true)

+--------+-----+
|    word|count|
+--------+-----+
|  아이브|    1|
|    가을|    1|
|아이해브|    1|
|  안유진|    1|
|    레이|    2|
|  장원영|    3|
|    리즈|    1|
|    이서|    2|
|  나오이|    1|
|    원영|    1|
+--------+-----+



In [7]:
# 1. 빈도수가 높은 순서대로 정렬하기 (Descending)
df.orderBy(df["count"].desc()).show()

# 2. 스파크 SQL 테이블처럼 조회하기 위해 임시 뷰(View) 생성
df.createOrReplaceTempView("words_table")

# 3. SQL 문법으로 쿼리 날리기
spark.sql("SELECT word, count FROM words_table WHERE count >= 2").show()

+--------+-----+
|    word|count|
+--------+-----+
|  장원영|    3|
|    레이|    2|
|    이서|    2|
|  아이브|    1|
|아이해브|    1|
|    가을|    1|
|  안유진|    1|
|    리즈|    1|
|  나오이|    1|
|    원영|    1|
+--------+-----+

+------+-----+
|  word|count|
+------+-----+
|  레이|    2|
|장원영|    3|
|  이서|    2|
+------+-----+



In [8]:
# 1. 간단한 텍스트 파일 생성 (노트북 환경 내부)
with open("sample.txt", "w") as f:
    f.write("Big Data Processing with Apache Spark\nSpark is lightning fast\nLove Spark")

# 2. 파일을 Spark DataFrame으로 읽어오기 (텍스트 파일)
text_df = spark.read.text("sample.txt")
text_df.show(truncate=False)

# 3. CSV나 Parquet 형식으로 저장해보기 (데이터프레임 저장)
# df.write.parquet("output_parquet")


+-------------------------------------+
|value                                |
+-------------------------------------+
|Big Data Processing with Apache Spark|
|Spark is lightning fast              |
|Love Spark                           |
+-------------------------------------+



In [9]:
# 1. 임의의 데이터프레임 만들기 (예: 직원 정보)
data = [
    ("Alice", "Engineering", 5000),
    ("Bob", "Marketing", 4500),
    ("Charlie", "Engineering", 6000),
    ("David", "Sales", 4000),
    ("Eve", "Marketing", 5500)
]

columns = ["name", "department", "salary"]
emp_df = spark.createDataFrame(data, columns)

# 2. SQL 쿼리를 날리기 위해 임시 뷰(View)로 등록
emp_df.createOrReplaceTempView("employees")

# 3. 순수 SQL 쿼리 실행하기
result_df = spark.sql("""
    SELECT department, AVG(salary) as avg_salary, COUNT(*) as emp_count
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""")

result_df.show()


+-----------+----------+---------+
| department|avg_salary|emp_count|
+-----------+----------+---------+
|Engineering|    5500.0|        2|
|  Marketing|    5000.0|        2|
|      Sales|    4000.0|        1|
+-----------+----------+---------+



In [10]:
# 1. 방금 만든 데이터프레임을 Parquet 포맷으로 디스크에 저장하기
emp_df.write.mode("overwrite").parquet("employees.parquet")

print("Parquet 파일 저장 완료!")

# 2. 저장된 Parquet 파일 다시 스파크 데이터프레임으로 읽어오기
loaded_df = spark.read.parquet("employees.parquet")

loaded_df.show()

Parquet 파일 저장 완료!
+-------+-----------+------+
|   name| department|salary|
+-------+-----------+------+
|Charlie|Engineering|  6000|
|  David|      Sales|  4000|
|    Eve|  Marketing|  5500|
|  Alice|Engineering|  5000|
|    Bob|  Marketing|  4500|
+-------+-----------+------+



In [11]:
# 1. 데이터프레임을 ORC 포맷으로 디스크에 저장하기
emp_df.write.mode("overwrite").orc("employees_orc")

print("ORC 파일 저장 완료!")

# 2. 저장된 ORC 파일 다시 스파크 데이터프레임으로 읽어오기
loaded_orc_df = spark.read.orc("employees_orc")

# 3. 데이터 확인
loaded_orc_df.show()


ORC 파일 저장 완료!
+-------+-----------+------+
|   name| department|salary|
+-------+-----------+------+
|Charlie|Engineering|  6000|
|  David|      Sales|  4000|
|    Eve|  Marketing|  5500|
|  Alice|Engineering|  5000|
|    Bob|  Marketing|  4500|
+-------+-----------+------+



In [12]:
# 파일 리스트나 메타데이터 구조를 확인하고 싶다면?
# (Hadoop 파일 시스템 API를 통해 폴더 안의 파일 확인)
sc = spark.sparkContext
conf = sc._jsc.hadoopConfiguration()
path = sc._jvm.org.apache.hadoop.fs.Path("employees_orc")
fs = path.getFileSystem(conf)

for f in fs.listStatus(path):
    print("저장된 파일:", f.getPath().getName())


저장된 파일: part-00001-86e10ce9-1dce-4468-bcaa-12728b806bda-c000.snappy.orc
저장된 파일: part-00000-86e10ce9-1dce-4468-bcaa-12728b806bda-c000.snappy.orc
저장된 파일: _SUCCESS


In [22]:
sc

<SparkContext master=local[*] appName=SparkTest>